# 04 — Document Type Classification

## Purpose
Identifies the document type and physical boundaries of each distinct
document within a parsed file. A single uploaded file may contain multiple
document types across its pages — this notebook detects those boundaries
and assigns a stable `CHILD_DOC_ID` to each segment, which becomes the
key all downstream steps (extraction, confidence scoring, review) operate
on.

## What this notebook does
Reads all pages from `DOCUMENTS_PAGES` for files with `STATUS = 'PARSED'`
that have not yet been classified, groups them by `DOC_ID`, and sends all
pages of each document together in a single `AI_COMPLETE` call. The model
returns a segment map identifying which pages belong to which document type
and where each distinct physical document begins and ends.

The classify model and prompt are loaded from `PIPELINE_CONFIG` at runtime
— no notebook edits are needed to change the model or tune the prompt.

Each segment in the response is assigned a `CHILD_DOC_ID`. For files
containing only one document type, `CHILD_DOC_ID` equals `DOC_ID`
— no split occurs. For files containing multiple document types or multiple
distinct documents of the same type, each segment gets a new UUID as its
`CHILD_DOC_ID`, with `DOC_ID` linking back to the original upload
for full lineage traceability.

`DOCUMENTS_PAGES` is then updated to tag each page with the `CHILD_DOC_ID`
of the segment it belongs to, so downstream steps can query pages by
segment without joining through `DOCUMENTS_CLASSIFIED`.

Status is updated in `DOCUMENTS_INGESTED` to `CLASSIFIED` on success or
`CLASSIFY_ERROR` on failure. The notebook is idempotent — already-classified
documents are excluded.

## Outputs
| Table | What is written |
|---|---|
| `PROCESSING.DOCUMENTS_CLASSIFIED` | One row per segment - `CHILD_DOC_ID`, `DOC_ID`, `DOC_TYPE`, page range, confidence, boundary signal |
| `PROCESSING.DOCUMENTS_PAGES` | `CHILD_DOC_ID` updated per page to reflect its segment assignment |
| `INGEST.DOCUMENTS_INGESTED` | STATUS updated to `CLASSIFIED` or `CLASSIFY_ERROR` |

## Key design decisions
- **One AI_COMPLETE call per file, not per page** — the model sees all
  pages together so it has full context when detecting boundaries between
  adjacent documents of the same type
- **Boundary signal captured** - the model returns a brief reason for each
  new segment boundary, stored in `BOUNDARY_SIGNAL` for reviewer visibility
  and debugging
- **Document description captured** - the model returns a brief description for each document
- **`CHILD_DOC_ID` is the downstream key** - all steps after classification
  operate on `CHILD_DOC_ID`, not `DOC_ID`. This cleanly separates the
  physical file identity (`DOC_ID`) from the logical document identity
  (`CHILD_DOC_ID`)
- **Config-driven** - model and prompt loaded from `PIPELINE_CONFIG` at
  runtime; no code change needed to swap models or tune the prompt

In [ ]:
import json
import re
import uuid
import pandas as pd
from collections import defaultdict
from snowflake.snowpark.context import get_active_session

DB                = 'PERMAFROST_POC'
INGEST_SCHEMA     = 'INGEST'
PROCESSING_SCHEMA = 'PROCESSING'
CONFIG_SCHEMA     = 'CONFIG'
AUDIT_SCHEMA      = 'AUDIT'

s = get_active_session()

def info(msg):    print(f"INFO:    {msg}")
def warning(msg): print(f"WARNING: {msg}")
def error(msg):   print(f"ERROR:   {msg}")

def estimate_tokens(text):
    return len(text) // 4 if text else 0

def parse_ai_response(raw):
    stripped = raw.strip()
    if stripped.startswith('"') and stripped.endswith('"'):
        stripped = json.loads(stripped)
    if '```' in stripped:
        parts   = stripped.split('```')
        content = parts[1]
        if content.startswith('json'):
            content = content[4:]
        stripped = content.strip()
    result = json.loads(stripped)
    if isinstance(result, str):
        result = json.loads(result)
    if not isinstance(result, dict):
        raise ValueError(f"Expected dict, got {type(result).__name__}")
    return result

def generate_unique_id(used_ids):
    while True:
        new_id = str(uuid.uuid4())
        if new_id not in used_ids:
            used_ids.add(new_id)
            return new_id

In [ ]:
#Load classify config
config = {
    row['CONFIG_KEY']: row['CONFIG_VALUE']
    for row in s.sql(f"""
        SELECT CONFIG_KEY, CONFIG_VALUE
        FROM {DB}.{CONFIG_SCHEMA}.PIPELINE_CONFIG
        WHERE CONFIG_KEY IN ('classify_model', 'classify_prompt')
        AND IS_ACTIVE = TRUE
    """).collect()
}

CLASSIFY_MODEL  = config.get('classify_model')
CLASSIFY_PROMPT = config.get('classify_prompt')

if not CLASSIFY_MODEL or not CLASSIFY_PROMPT:
    raise ValueError(
        "Missing classify config in PIPELINE_CONFIG - "
        "run 00_setup_config.ipynb first"
    )

info(f"Classify model : {CLASSIFY_MODEL}")
info(f"Classify prompt: {len(CLASSIFY_PROMPT)} chars")

In [ ]:
# Build classify queue via 
# LISTAGG concatenates all pages per DOC_ID
# No Python grouping or threading needed

queue_rows = s.sql(f"""
    SELECT
        p.DOC_ID,
        LISTAGG(
            '[PAGE ' || p.PAGE_NUMBER || ']' || CHR(10) ||
            COALESCE(p.PAGE_CONTENT_TRANSLATED, p.PAGE_CONTENT),
            '\\n\\n'
        ) WITHIN GROUP (ORDER BY p.PAGE_NUMBER)    AS PAGE_BLOCK,
        SUM(LENGTH(COALESCE(
            p.PAGE_CONTENT_TRANSLATED, p.PAGE_CONTENT
        )))                                         AS TOTAL_CHARS
    FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_PAGES p
    JOIN {DB}.{INGEST_SCHEMA}.DOCUMENTS_INGESTED i
        ON p.DOC_ID = i.DOC_ID
    LEFT JOIN {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_CLASSIFIED c
        ON p.DOC_ID = c.DOC_ID
    WHERE i.STATUS      = 'PARSED'
      AND c.DOC_ID      IS NULL
      AND p.PAGE_CONTENT IS NOT NULL
    GROUP BY p.DOC_ID
    ORDER BY p.DOC_ID
""").collect()

info(f"Classify queue: {len(queue_rows)} document(s)")

if not queue_rows:
    print("\nNothing to classify.")

In [ ]:
prompt_prefix= CLASSIFY_PROMPT.split('{pages}')[0]

classified_rows = s.sql(f"""
    SELECT
        p.DOC_ID,
        LISTAGG(
            '[PAGE ' || p.PAGE_NUMBER || ']' || CHR(10) ||
            COALESCE(p.PAGE_CONTENT_TRANSLATED, p.PAGE_CONTENT),
            '\\n\\n'
        ) WITHIN GROUP (ORDER BY p.PAGE_NUMBER)      AS PAGE_BLOCK,
        SUM(LENGTH(COALESCE(
            p.PAGE_CONTENT_TRANSLATED, p.PAGE_CONTENT
        )))                                           AS TOTAL_CHARS,
        AI_COMPLETE(
            ?,
            ? || LISTAGG(
                    '[PAGE ' || p.PAGE_NUMBER || ']' || CHR(10) ||
                    COALESCE(p.PAGE_CONTENT_TRANSLATED, p.PAGE_CONTENT),
                    '\\n\\n'
                 ) WITHIN GROUP (ORDER BY p.PAGE_NUMBER)
        )                                               AS RAW_RESPONSE
    FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_PAGES p
    JOIN {DB}.{INGEST_SCHEMA}.DOCUMENTS_INGESTED i
        ON p.DOC_ID = i.DOC_ID
    LEFT JOIN {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_CLASSIFIED c
        ON p.DOC_ID = c.DOC_ID
    WHERE i.STATUS      = 'PARSED'
      AND c.DOC_ID      IS NULL
      AND p.PAGE_CONTENT IS NOT NULL
    GROUP BY p.DOC_ID
    ORDER BY p.DOC_ID
""", params=[CLASSIFY_MODEL, prompt_prefix]).collect()

info(f"Classify queue: {len(classified_rows)} document(s)")

if not classified_rows:
    print("\nNothing to classify.")

# Parse results in Python, per row
for row in classified_rows:
    doc_id      = row['DOC_ID']
    page_block  = row['PAGE_BLOCK']
    raw         = row['RAW_RESPONSE']

    if not page_block or not page_block.strip():
        warning(f"  [SKIP] {doc_id} - empty page content")
        continue

    try:
        result   = parse_ai_response(raw)
        segments = result.get('segments', [])

        if not segments:
            raise ValueError("AI_COMPLETE returned empty segments array")

        prompt        = prompt_prefix + page_block + prompt_suffix
        input_tokens  = estimate_tokens(prompt)
        output_tokens = estimate_tokens(raw)

        llm_rows.append({
            'DOC_ID':        doc_id,
            'CHILD_DOC_ID':  None,
            'PIPELINE_STEP': 'CLASSIFY',
            'MODEL_NAME':    CLASSIFY_MODEL,
            'TOKENS_IN':     input_tokens,
            'TOKENS_OUT':    output_tokens,
        })

        classify_results.append({'doc_id': doc_id, 'segments': segments})

        seg_summary = ', '.join(
            f"{seg['doc_type']} pp.{seg['page_start']}-{seg['page_end']}"
            for seg in segments
        )
        info(f"  [OK] {doc_id} - {seg_summary}")

    except Exception as e:
        classify_errors.append({'doc_id': doc_id, 'error': str(e)})
        error(f"  [FAIL] {doc_id}: {e}")

In [ ]:
# Build DOCUMENTS_CLASSIFIED rows and DOCUMENTS_PAGES updates

classified_rows = []
page_updates    = []
all_used_ids    = set(
    row['CHILD_DOC_ID']
    for row in s.sql(f"""
        SELECT CHILD_DOC_ID FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_CLASSIFIED
    """).collect()
) | set(
    row['DOC_ID']
    for row in s.sql(f"""
        SELECT DOC_ID FROM {DB}.{INGEST_SCHEMA}.DOCUMENTS_INGESTED
    """).collect()
)


for result in classify_results:
    doc_id   = result['doc_id']
    segments = result['segments']

    for seg in segments:
        child_doc_id = doc_id if len(segments) == 1 \
                       else generate_unique_id(all_used_ids)

        classified_rows.append({
            'CHILD_DOC_ID':        child_doc_id,
            'DOC_ID':              doc_id,
            'DOC_TYPE':            seg.get('doc_type'),
            'PAGE_START':          seg.get('page_start'),
            'PAGE_END':            seg.get('page_end'),
            'CONFIDENCE':          seg.get('confidence'),
            'BOUNDARY_SIGNAL':     seg.get('boundary_signal'),
            'DOCUMENT_DESCRIPTION': seg.get('document_description'), 
        })

        page_updates.append({
            'child_doc_id': child_doc_id,
            'doc_id':       doc_id,
            'page_start':   seg.get('page_start'),
            'page_end':     seg.get('page_end'),
        })

# Write DOCUMENTS_CLASSIFIED 
if classified_rows:
    s.write_pandas(
        pd.DataFrame(classified_rows),
        table_name='DOCUMENTS_CLASSIFIED',
        database=DB, schema=PROCESSING_SCHEMA,
        overwrite=False,
    )
    info(f"Wrote {len(classified_rows)} row(s) to DOCUMENTS_CLASSIFIED")

# Update CHILD_DOC_ID on DOCUMENTS_PAGES — bulk via staging 
if page_updates:
    s.write_pandas(
        pd.DataFrame([{
            'CHILD_DOC_ID': upd['child_doc_id'],
            'DOC_ID':       upd['doc_id'],
            'PAGE_START':   upd['page_start'],
            'PAGE_END':     upd['page_end'],
        } for upd in page_updates]),
        table_name='PAGE_UPDATE_STAGING',
        database=DB, schema=PROCESSING_SCHEMA,
        overwrite=True,
        auto_create_table=True,
    )

    s.sql(f"""
        UPDATE {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_PAGES p
        SET p.CHILD_DOC_ID = t.CHILD_DOC_ID
        FROM {DB}.{PROCESSING_SCHEMA}.PAGE_UPDATE_STAGING t
        WHERE p.DOC_ID       = t.DOC_ID
          AND p.PAGE_NUMBER BETWEEN t.PAGE_START AND t.PAGE_END
    """).collect()

    s.sql(f"""
        DROP TABLE IF EXISTS {DB}.{PROCESSING_SCHEMA}.PAGE_UPDATE_STAGING
    """).collect()

    info(f"Updated CHILD_DOC_ID on {len(page_updates)} segment(s) in DOCUMENTS_PAGES")

# Write LLM_USAGE
if llm_rows:
    s.write_pandas(
        pd.DataFrame(llm_rows),
        table_name='LLM_USAGE',
        database=DB, schema=AUDIT_SCHEMA,
        overwrite=False,
    )
    info(f"Wrote {len(llm_rows)} LLM_USAGE row(s)")

# Update STATUS in DOCUMENTS_INGESTED
if classify_results:
    classified_ids = [r['doc_id'] for r in classify_results]
    id_list = ','.join(f"'{d}'" for d in classified_ids)
    s.sql(f"""
        UPDATE {DB}.{INGEST_SCHEMA}.DOCUMENTS_INGESTED
        SET STATUS = 'CLASSIFIED'
        WHERE DOC_ID IN ({id_list})
    """).collect()

if classify_errors:
    error_ids = [e['doc_id'] for e in classify_errors]
    id_list = ','.join(f"'{d}'" for d in error_ids)
    s.sql(f"""
        UPDATE {DB}.{INGEST_SCHEMA}.DOCUMENTS_INGESTED
        SET STATUS = 'CLASSIFY_ERROR'
        WHERE DOC_ID IN ({id_list})
    """).collect()

In [ ]:
# Summary 
print(f"\n Classification summary ")
print(f"  Classified successfully : {len(classify_results)}")
print(f"  Segments created        : {len(classified_rows)}")
print(f"  Errors                  : {len(classify_errors)}")
print(f"  Total input tokens      : {sum(r['TOKENS_IN'] for r in llm_rows):,}")
print(f"  Total output tokens     : {sum(r['TOKENS_OUT'] for r in llm_rows):,}")

if classify_errors:
    print("\n  Failed documents:")
    for e in classify_errors:
        print(f"    {e['doc_id']}: {e['error']}")

print(f"\n DOCUMENTS_CLASSIFIED breakdown")
s.sql(f"""
    SELECT
        DOC_TYPE,
        COUNT(*)                        AS SEGMENT_COUNT,
        ROUND(AVG(CONFIDENCE), 3)       AS AVG_CONFIDENCE,
        COUNT(DOCUMENT_DESCRIPTION)     AS WITH_DESCRIPTION
    FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_CLASSIFIED
    GROUP BY DOC_TYPE
    ORDER BY SEGMENT_COUNT DESC
""").show()

print(f"\n Sample document descriptions")
s.sql(f"""
    SELECT
        DOC_TYPE,
        CHILD_DOC_ID,
        DOCUMENT_DESCRIPTION
    FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_CLASSIFIED
    WHERE DOCUMENT_DESCRIPTION IS NOT NULL
    ORDER BY DOC_TYPE, CHILD_DOC_ID
    LIMIT 10
""").show()